In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Read the CSV file
df = pd.read_csv("../data/end-to-end/report_complete_processed.csv")

# Filter for meta_202210_kv trace and remove LAMA and marginal-hits-tuned
df = df[(df['trace_name'] == 'meta_202210_kv') & 
        (df['rebalance_strategy'] != 'lama') &
        (df['rebalance_strategy'] != 'marginal-hits-tuned')]

print(f"Data shape after filtering: {df.shape}")
print(f"Available strategies: {sorted(df['rebalance_strategy'].unique())}")
print(f"Available allocators: {sorted(df['allocator'].unique())}")
print(f"WSR range: {df['wsr'].min():.3f} - {df['wsr'].max():.3f}")

Data shape after filtering: (140, 26)
Available strategies: ['disabled', 'eviction-rate', 'hits', 'marginal-hits', 'tail-age']
Available allocators: ['LRU', 'LRU2Q', 'TINYLFU']
WSR range: 0.004 - 0.400


In [6]:
# Define strategy order and labels (same as matplotlib version)
strategy_order = ["disabled", "tail-age", "eviction-rate", "hits", "marginal-hits"]
strategy_labels = {
    "disabled": "Disabled",
    "tail-age": "Tail-Age", 
    "eviction-rate": "Eviction-Rate",
    "hits": "Hits-Per-Slab",
    "marginal-hits": "Marginal-Hits"
}

# Define allocator order and labels
allocator_order = ['LRU', 'LRU2Q', 'TINYLFU']
allocator_labels = ['LRU', 'TwoQ', 'TinyLFU']

# Prepare data for plotting
df['wsr_percent'] = df['wsr'] * 100
df['strategy_label'] = df['rebalance_strategy'].map(strategy_labels)
df['allocator_label'] = df['allocator'].map(dict(zip(allocator_order, allocator_labels)))

# Create categorical ordering for consistent plotting
df['strategy_cat'] = pd.Categorical(df['rebalance_strategy'], categories=strategy_order, ordered=True)
df['allocator_cat'] = pd.Categorical(df['allocator'], categories=allocator_order, ordered=True)

# Sort data for proper line plotting
df_sorted = df.sort_values(['allocator_cat', 'strategy_cat', 'wsr_percent'])

print("Data prepared for plotting!")
print(f"Strategy order: {strategy_order}")
print(f"Allocator order: {allocator_order}")

Data prepared for plotting!
Strategy order: ['disabled', 'tail-age', 'eviction-rate', 'hits', 'marginal-hits']
Allocator order: ['LRU', 'LRU2Q', 'TINYLFU']


In [7]:
# Create the interactive plot
fig = go.Figure()

# Get unique WSR values and sort them for categorical x-axis
unique_wsr = sorted(df['wsr'].unique())
wsr_labels = [f"{wsr*100:.1f}" for wsr in unique_wsr]
x_positions = list(range(len(unique_wsr)))

# Define line styles for allocators (Plotly dash patterns)
allocator_dash = {
    'LRU': 'solid',
    'LRU2Q': 'dash', 
    'TINYLFU': 'dashdot'
}

# Define symbols for allocators
allocator_symbol = {
    'LRU': 'circle',
    'LRU2Q': 'square',
    'TINYLFU': 'triangle-up'
}

# Get default Plotly colors for strategies (enforcing order)
colors = px.colors.qualitative.Plotly
strategy_colors = {strategy: colors[i % len(colors)] for i, strategy in enumerate(strategy_order)}

# Plot lines for each strategy-allocator combination
for i, strategy in enumerate(strategy_order):
    if strategy not in df['rebalance_strategy'].values:
        continue
        
    for j, allocator in enumerate(allocator_order):
        if allocator not in df['allocator'].values:
            continue
            
        # Filter data for this strategy-allocator combination
        subset = df_sorted[(df_sorted['rebalance_strategy'] == strategy) & 
                          (df_sorted['allocator'] == allocator)]
        
        if subset.empty:
            continue
        
        # Map WSR values to categorical positions
        x_values = [x_positions[unique_wsr.index(wsr)] for wsr in subset['wsr']]
        y_values = subset['miss_ratio']
        
        # Create trace name for legend
        trace_name = f"{strategy_labels[strategy]} + {allocator_labels[j]}"
        
        # Add trace
        fig.add_trace(go.Scatter(
            x=x_values,
            y=y_values,
            mode='lines+markers',
            name=trace_name,
            line=dict(
                color=strategy_colors[strategy],
                dash=allocator_dash[allocator],
                width=3
            ),
            marker=dict(
                symbol=allocator_symbol[allocator],
                size=10,
                color=strategy_colors[strategy],
                line=dict(color='white', width=1)
            ),
            legendgroup=strategy,  # Group by strategy for better legend organization
            showlegend=True
        ))

print("Plot traces added successfully!")

Plot traces added successfully!


In [8]:
# Update layout for publication quality
fig.update_layout(
    title=dict(
        text="Meta KV Trace: Miss Ratio vs Cache Size",
        font=dict(size=24),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title="Cache Size (% of Working Set)",
        title_font=dict(size=20),
        tickfont=dict(size=18),
        tickmode='array',
        tickvals=x_positions,
        ticktext=wsr_labels,
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray'
    ),
    yaxis=dict(
        title="Miss Ratio",
        title_font=dict(size=20),
        tickfont=dict(size=18),
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray'
    ),
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left", 
        x=1.02,
        font=dict(size=14),
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="black",
        borderwidth=1
    ),
    width=1000,
    height=600,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(r=250)  # Extra space for legend on the right
)

# Show the interactive plot
fig.show()

print("Interactive Plotly figure created!")

Interactive Plotly figure created!


In [ ]:
# Optional: Save the plot as HTML
fig.write_html("meta_kv_plotly_interactive.html")
print("Plot saved as: meta_kv_plotly_interactive.html")

# Optional: Save as static image (requires kaleido: pip install kaleido)
try:
    fig.write_image("meta_kv_plotly_static.pdf", width=1200, height=800)
    print("Plot saved as: meta_kv_plotly_static.pdf")
except Exception as e:
    print(f"Could not save static image: {e}")
    print("To save static images, install kaleido: pip install kaleido")

In [ ]:
# Alternative version: Create plot with better legend organization
# This version groups traces by strategy first, then by allocator

fig2 = go.Figure()

# Create traces with better legend grouping
for i, strategy in enumerate(strategy_order):
    if strategy not in df['rebalance_strategy'].values:
        continue
    
    strategy_color = strategy_colors[strategy]
    
    for j, allocator in enumerate(allocator_order):
        if allocator not in df['allocator'].values:
            continue
            
        # Filter data for this strategy-allocator combination
        subset = df_sorted[(df_sorted['rebalance_strategy'] == strategy) & 
                          (df_sorted['allocator'] == allocator)]
        
        if subset.empty:
            continue
        
        # Map WSR values to categorical positions
        x_values = [x_positions[unique_wsr.index(wsr)] for wsr in subset['wsr']]
        y_values = subset['miss_ratio']
        
        # Create trace name showing strategy first, then allocator
        trace_name = f"{strategy_labels[strategy]} ({allocator_labels[j]})"
        
        # Add trace
        fig2.add_trace(go.Scatter(
            x=x_values,
            y=y_values,
            mode='lines+markers',
            name=trace_name,
            line=dict(
                color=strategy_color,
                dash=allocator_dash[allocator],
                width=3
            ),
            marker=dict(
                symbol=allocator_symbol[allocator],
                size=10,
                color=strategy_color,
                line=dict(color='white', width=1)
            ),
            legendgroup=f"group_{i}",  # Group by strategy index for ordering
            showlegend=True
        ))

# Update layout (similar to first version)
fig2.update_layout(
    title=dict(
        text="Meta KV Trace: Miss Ratio vs Cache Size (Grouped Legend)",
        font=dict(size=24),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title="Cache Size (% of Working Set)",
        title_font=dict(size=20),
        tickfont=dict(size=18),
        tickmode='array',
        tickvals=x_positions,
        ticktext=wsr_labels,
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray'
    ),
    yaxis=dict(
        title="Miss Ratio",
        title_font=dict(size=20),
        tickfont=dict(size=18),
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray'
    ),
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left", 
        x=1.02,
        font=dict(size=14),
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="black",
        borderwidth=1,
        traceorder="grouped"  # Keep traces grouped by legendgroup
    ),
    width=1000,
    height=600,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(r=300)  # Extra space for longer legend labels
)

fig2.show()
print("Alternative version with grouped legend created!")